<a href="https://colab.research.google.com/github/ireneminhee/BKBS/blob/news_bind/Similarity_based_recommendation_and_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
from sentence_transformers import SentenceTransformer, util

In [5]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
article = pd.read_csv("/content/drive/MyDrive/BKBS/news_bind.csv")
article = article.drop_duplicates(subset=["title", "source"])
article

,context_id,title,source,published,code,context
0,464098,"익산 서부권역 다목적 체육관, 다음달 개관",전북일보,2021-03-25,스포츠,익산시 서부권역 다목적 체육관이 내달 개관을 앞두고 막바지 개관 준비가 한창이다. ...
1,464131,"무주군건강가정·다문화가족지원센터, '아빠하고 나하고' 프로그램 운영",전북일보,2021-03-25,지역,무주군건강가정ㆍ다문화가족지원센터(센터장 장진원)가 아버지와 자녀를 대상으로 ‘아빠하...
2,464132,양희은의 '뜻밖의 만남',전북일보,2021-03-25,문화,양희은의 노래 <엄마가 딸에게>는 엄마와 딸이 차마 말하지 못하는 속마음을 서로에게...
3,464133,전북 산업단지 대개조 지역경제 도약 기회로,전북일보,2021-03-25,IT과학,"군산ㆍ익산ㆍ완주지역의 노후 산업단지에 내년부터 3년간 국비와 지방비, 민간자본 등 ..."
4,464134,익산 함라면 야산서 불… 신원 미상 남성 시신 1구 발견,전북일보,2021-03-25,사회,25일 오전 11시 35분께 익산시 함라면의 한 야산에서 불이 나 1시간 30분만에...
...,...,...,...,...,...,...
136011,1121886,“5월9일 밤∼10일 새벽 부천 ‘메리트나이트’ 방문자는 코로나19 검사 요청”,세계일보,2020-05-18,사회,이태원 클럽 방문 확진자 1명이 감염력이 있는 기간에 부천 지역 유흥시설을 방문한 ...
136012,1121892,"'이태원 클럽' 확진자, 부천 나이트클럽도 방문…지역감염 확산 우려",세계일보,2020-05-18,사회,이태원 클럽을 다녀온 뒤 신종 코로나바이러스 감염증(코로나19)에 감염된 환자 중 ...
136013,1121893,"동국제약, 코로나19 진정 효과 ‘포폴주사’ 주요 4개국 수출",세계일보,2020-05-18,IT과학,"동국제약(대표이사 오흥주)은 네덜란드, 룩셈부르크, 싱가폴, 일본 등 유럽과 아시아..."
136014,1121895,"송가인, 5·18 민주화운동 희생자 추모 “영원히 잊지 않을 것”",세계일보,2020-05-18,지역,트로트 가수 송가인이 5·18 민주화운동 희생자들을 추모했다. \n\n송가인은 18...


In [7]:
article.info()

<class 'pandas.core.frame.DataFrame'>
Index: 135785 entries, 0 to 136015
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   context_id  135785 non-null  int64 
 1   title       135785 non-null  object
 2   source      135785 non-null  object
 3   published   135785 non-null  object
 4   code        135785 non-null  object
 5   context     135785 non-null  object
dtypes: int64(1), object(5)
memory usage: 7.3+ MB


## 추천

In [8]:
def recommend_articles_by_cosine(base_article, articles_df, model, top_n=5, time_window=30):
    # 1. published 컬럼 datetime 변환
    articles_df = articles_df.copy()
    articles_df['published'] = pd.to_datetime(articles_df['published'], format="%Y-%m-%d")

    # base_article 날짜도 datetime으로 변환
    base_article = base_article.copy()
    base_article['published'] = pd.to_datetime(base_article['published'], format="%Y-%m-%d")

    # 분석에 필요한 컬럼만 유지
    articles_df = articles_df.loc[:, ['context_id', 'title', 'context', 'published', 'source']]

    # 2. 후보 기사 필터링 (출판일 기준)
    start_date = base_article['published'] - timedelta(days=time_window)
    end_date = base_article['published'] + timedelta(days=time_window)

    candidate_articles = articles_df[
        (articles_df['published'] >= start_date) &
        (articles_df['published'] <= end_date)
    ].reset_index(drop=True)

    # 기준 기사 제외
    if 'context_id' in articles_df.columns and 'context_id' in base_article:
        candidate_articles = candidate_articles[candidate_articles['context_id'] != base_article['context_id']]
    else:
        candidate_articles = candidate_articles[candidate_articles['title'] != base_article['title']]

    if candidate_articles.empty:
        return pd.DataFrame(columns=['recommend_id', 'title', 'context', 'published', 'source', 'similarity', 'base_id'])

    # 3. 텍스트 임베딩
    model = SentenceTransformer('all-MiniLM-L6-v2')

    base_text = base_article['title'] + " " + base_article['context']
    base_embedding = model.encode(base_text, convert_to_tensor=True)

    candidate_texts = candidate_articles['title'] + " " + candidate_articles['context']
    candidate_embeddings = model.encode(candidate_texts.tolist(), convert_to_tensor=True)

    # 4. 유사도 계산
    cos_scores = util.pytorch_cos_sim(base_embedding, candidate_embeddings)[0]
    candidate_articles['similarity'] = cos_scores.cpu().numpy()

    # 5. 상위 N개 추천
    top_articles = candidate_articles.sort_values(by='similarity', ascending=False).head(top_n)

    # base_id, recommend_id 추가
    top_articles['base_id'] = base_article['context_id']
    top_articles['recommend_id'] = top_articles['context_id']

    return top_articles[['recommend_id', 'title', 'context', 'published', 'source', 'similarity', 'base_id']]


In [9]:
base_article = [
    {
        "context_id": article[article['title'].str.contains("배터리 자립")]['context_id'].item(),
        "title": article[article['title'].str.contains("배터리 자립")]['title'].item(),
        "context": article[article['title'].str.contains("배터리 자립")]['context'].item(),
        "published": article[article['title'].str.contains("배터리 자립")]['published'].item(),
        "source": article[article['title'].str.contains("배터리 자립")]['source'].item()
    },
    {
        "context_id": article[article['title'].str.contains("LH투기 후폭풍, LH 타사업지 사업 차질 현실화")]['context_id'].item(),
        "title": article[article['title'].str.contains("LH투기 후폭풍, LH 타사업지 사업 차질 현실화")]['title'].item(),
        "context": article[article['title'].str.contains("LH투기 후폭풍, LH 타사업지 사업 차질 현실화")]['context'].item(),
        "published": article[article['title'].str.contains("LH투기 후폭풍, LH 타사업지 사업 차질 현실화")]['published'].item(),
        "source": article[article['title'].str.contains("LH투기 후폭풍, LH 타사업지 사업 차질 현실화")]['source'].item()
    },
    {
        "context_id": article[article['title'].str.contains("대선 앞두고 ‘中 심판론’ 고삐 죄는 트럼프")]['context_id'].item(),
        "title": article[article['title'].str.contains("대선 앞두고 ‘中 심판론’ 고삐 죄는 트럼프")]['title'].item(),
        "context": article[article['title'].str.contains("대선 앞두고 ‘中 심판론’ 고삐 죄는 트럼프")]['context'].item(),
        "published": article[article['title'].str.contains("대선 앞두고 ‘中 심판론’ 고삐 죄는 트럼프")]['published'].item(),
        "source": article[article['title'].str.contains("대선 앞두고 ‘中 심판론’ 고삐 죄는 트럼프")]['source'].item()
    }
]
base_article

[{'context_id': 574954,
  'title': '배터리 자립 선언하는 車업계… “어느 세월에” 속내는 착잡',
  'context': '미국 자동차 시장 점유율 1위 포드의 최고경영자(CEO) 짐 팔리는 최근 한 포럼에서 전기차 시장에서 살아남기 위해 ‘배터리 내재화’가 필요하다고 강조했다. 그로부터 약 일주일 뒤 곧바로 배터리 자체 생산 계획을 공식화했다. 디트로이트에 1억8500만 달러(2060억원)를 들여 배터리 연구·개발센터인 ‘포드이온파크’를 설립하고, 리튬이온 배터리를 자체 개발할 예정이다. \n \n글로벌 완성차 업계의 ‘배터리 권력’ 쟁탈전이 뜨겁다. 폭스바겐, 제너럴모터스(GM)에 이어 포드까지 배터리 내재화를 공식 선언했다. \n \n배터리 자체 생산으로 원가 절감 효과를 보지 못한다면 미래 전기차 시장에서 도태될 수 있다는 위기감이 반영된 결과다. \n \n다만 업계의 무게 중심이 내연기관 엔진에서 배터리와 반도체로 개편되고 있다는 점이 부담이다. ‘전기차의 심장’이라 불리는 배터리가 협력 업체에서 개발이 이뤄진다면 완성차 업체가 기대하는대로 기존의 원·하청 수직 관계가 유지되기 어렵다는 관측이 나오고 있어서다. 전기차 배터리는 장기간 개발이 이뤄져야 양산이 가능하다는 점도 리스크 요인으로 꼽힌다. \n \n세계 1위 완성차 업체인 폭스바겐은 최근 2030년까지 유럽 내 배터리 공장 6곳을 증설해 연간 240GWh 배터리 셀을 만들겠다고 밝혔다. 블룸버그에 따르면 유럽의 전기차 배터리 관련 지원금은 1년 사이 10배 증가한 61억 유로(8조900억원)에 이른다. \n \nGM은 자체 생산보다는 배터리 전문 업체인 LG에너지솔루션과 협업을 선택했다. LG와의 합작법인인 ‘얼티엄셀즈’를 설립하고 미 오하이오주에 연 30GWh 규모의 배터리 공장을 짓고 있다. 향후 테네시주에도 두 번째 공장을 들이기로 했다. 테슬라와 현대자동차 역시 배터리 자체 생산을 추진하겠다고 발표한 바 있다. \n \n배터리 내재화 바람은 전기차 배터리의 

In [11]:
recommend_articles = pd.DataFrame()

for i in range(len(base_article)):
    top_articles = recommend_articles_by_cosine(base_article[i], article, model)
    recommend_articles = pd.concat([recommend_articles, top_articles])

recommend_articles

,recommend_id,title,context,published,source,similarity,base_id
33717,671152,글로벌 기업도 못 피해가는 불매운동... 소비자는 '진심'을 원한다,불매운동은 글로벌 기업들도 피해가지 못한다. 올해 1분기 세계 전기차 시장 판매량 ...,2021-05-31,한국일보,0.871962,574954
29897,664892,"'車반도체 품귀 막자'... 홍남기 ""차량용 반도체 예산, 내년 대폭 늘린다""",정부가 차량용 반도체 수급 부족 문제를 해결하기 위해 단기간 내 사업화할 수 있는 ...,2021-04-16,한국일보,0.862631,574954
37978,577353,“에너지 사용 미쳤다” 머스크 또 트윗…도지코인 띄우기?,테슬라 최고경영자(CEO) 일론 머스크가 비트코인 결제 중단 선언에 이어 영국의 한...,2021-05-14,국민일보,0.861510,574954
30506,665718,"파나소닉, 테슬라 이외의 배터리 사업 다각화 강조","세계 최대 EV 배터리 생산 업체 중 하나인 파나소닉의 CEO, 쓰가 가즈히로(Ts...",2021-04-14,한국일보,0.857370,574954
36443,562138,"‘쓰미마셍, 우리도 없어요~’ 일본도 뜻밖의 車반도체 난리",정부가 최근 일본의 반도체 제조사 르네사스에 차량용 반도체 지원 요청을 한 것으로 ...,2021-06-02,국민일보,0.854153,574954
88,464227,"LH 관련 익산 현안사업, 차질 없이 ‘순항’","LH 투기 의혹 후폭풍이 전국을 강타하고 있는 가운데, LH 관련 익산 현안사업은 ...",2021-03-23,전북일보,0.951278,510996
32188,652525,"정세균, 변창흠 LH 직원 옹호 발언에 ""적절치 않았다"" 비판",정세균 국무총리가 10일 한국토지주택공사(LH) 직원들의 투기 의혹과 관련해 여권에...,2021-03-10,한국일보,0.943926,510996
44960,604689,“LH 임직원 사전투기 원천차단 위해 투기이익 환수해야”,한국토지주택공사(LH) 직원의 광명·시흥 신도시 사전투기 의혹이 일파만파로 확산하고...,2021-03-04,부산일보,0.943238,510996
1742,466204,새만금개발공사로 옮겨 붙은 ‘LH발 부동산 투기 의혹’,한국토지주택공사(LH) 발 부동산 투기 의혹이 새만금개발공사로 옮겨 붙었다. \n\...,2021-03-28,전북일보,0.942947,510996
47720,604551,LH 직원 ‘토지경매 1타 강사’ 홍보하며 유료사이트 강의,한국토지주택공사(LH)의 한 직원이 부동산 유료 강의사이트에서 ‘토지 경매 1타 강...,2021-03-04,부산일보,0.942124,510996


## 평가

In [12]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# 높을수록 기사 내용이 다양
def diversity_score(recommend_articles, model):
    texts = (recommend_articles['title'] + " " + recommend_articles['context']).tolist()
    if len(texts) < 2:
        return 0.0

    embeddings = model.encode(texts, convert_to_tensor=False)  # numpy array
    sim_matrix = cosine_similarity(embeddings)

    n = len(sim_matrix)
    upper = sim_matrix[np.triu_indices(n, k=1)]
    avg_sim = np.mean(upper)

    return (1 - avg_sim).round(4)

In [13]:
# 높을수록 추천 언론사 분포가 균형
def cgi_score(recommend_articles):
    # 언론사 리스트
    publisher_list = recommend_articles['source'].tolist()
    if len(publisher_list) < 2:
        return 0.0

    # 언론사별 개수
    counts = np.array(list({p: publisher_list.count(p) for p in set(publisher_list)}.values()))
    counts = np.sort(counts)  # 오름차순 정렬
    n = len(counts)           # 언론사 개수

    cum_counts = np.cumsum(counts)
    S_n = cum_counts[-1]      # 총 추천 개수
    numerator = 2 * np.sum(cum_counts[:-1])
    denominator = n * S_n - S_n  # n(n-1)\bar{c} 과 동일
    gini = numerator / denominator if denominator != 0 else 0.0

    return (1 - gini).round(4)   # 공식상 CGI = 1 - Gini

In [14]:
def per_seed_scores(df, model, seed_col='seed_article_id'):
    """
    각 seed_article(원문)별로 diversity_score, cgi_score 계산.
    반환: DataFrame with columns [seed, diversity, cgi, n_recommendations]
    """
    rows = []
    for seed, group in df.groupby(seed_col):
        texts_count = len(group)
        if texts_count < 2:
            # 추천이 0~1개면 지표 안정성 없으므로 NaN 처리 (혹은 0으로 처리 가능)
            rows.append((seed, np.nan, np.nan, texts_count))
            continue
        d = diversity_score(group, model)
        c = cgi_score(group)
        rows.append((seed, d, c, texts_count))
    return pd.DataFrame(rows, columns=[seed_col, 'diversity', 'cgi', 'n_recommendations'])


# === 부트스트랩으로 전체 평균의 신뢰구간을 구하는 함수 ===
def bootstrap_mean_ci(values, n_boot=1000, ci=0.95, random_state=None):
    """
    values: 1d array-like (NaN 포함 가능 -> 제거)
    returns: dict {'mean':..., 'std':..., 'ci_low':..., 'ci_high':..., 'samples':n_non_na}
    """
    vals = np.array(values)
    vals = vals[~np.isnan(vals)]
    n = len(vals)
    if n == 0:
        return {'mean': np.nan, 'std': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n': 0}
    rng = np.random.RandomState(random_state)
    boot_means = []
    for _ in range(n_boot):
        sample = rng.choice(vals, size=n, replace=True)
        boot_means.append(np.mean(sample))

    mean = np.mean(vals).round(4)
    std = np.std(vals, ddof=1).round(4)
    alpha = 1 - ci
    low = np.percentile(boot_means, 100 * (alpha / 2)).round(4)
    high = np.percentile(boot_means, 100 * (1 - alpha / 2)).round(4)
    return {'mean': mean, 'std': std, 'ci_low': low, 'ci_high': high, 'n': n}


def aggregate_diversity_report(df, model, seed_col='seed_article_id', n_boot=2000):
    per_seed = per_seed_scores(df, model, seed_col=seed_col)

    # per-seed 분포(결측 제거)
    div_values = per_seed['diversity'].values
    cgi_values = per_seed['cgi'].values

    div_stats = bootstrap_mean_ci(div_values, n_boot=n_boot, ci=0.95, random_state=42)
    cgi_stats = bootstrap_mean_ci(cgi_values, n_boot=n_boot, ci=0.95, random_state=43)

    report = {
        'per_seed_df': per_seed,
        'diversity': div_stats,
        'cgi': cgi_stats
    }
    return report


In [15]:
report = aggregate_diversity_report(recommend_articles, model, seed_col="base_id", n_boot=2000)
print(report)

{'per_seed_df':    base_id  diversity  cgi  n_recommendations
0   510996     0.0438  0.2                  5
1   574954     0.1941  0.2                  5
2  1121877     0.0757  0.4                  5, 'diversity': {'mean': np.float32(0.1045), 'std': np.float32(0.0792), 'ci_low': np.float32(0.0438), 'ci_high': np.float32(0.1941), 'n': 3}, 'cgi': {'mean': np.float64(0.2667), 'std': np.float64(0.1155), 'ci_low': np.float64(0.2), 'ci_high': np.float64(0.4), 'n': 3}}


In [18]:
!jupyter nbconvert --ClearMetadataPreprocessor.enabled=True --to notebook --inplace "유사도 기반 추천 _및_평가.ipynb"


[NbConvertApp] WARNING | pattern '유사도 기반 추천 _및_평가.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute